# MAGPAI Tiny ANN Training Notebook v1.0

**Artifact Name:** MAGPAI Tiny ANN Training Notebook  
**Version:** 1.0  
**Date:** 2026-06-01  
**Author:** javaboy-vk  

## Purpose

This notebook demonstrates the MAGPAI tiny ANN concept we discussed:

> The question stays fixed: **“Are MAG sales up in Chicago?”**  
> The training examples change.  
> The weights and bias change.  
> The answer/confidence changes.

This notebook is meant to teach:

1. What the input tensor/vector represents.
2. What weights are.
3. What bias is.
4. How training changes weights and bias.
5. How MAGPAI re-answers the same question after each training example.


## 1. The Core Mental Model

MAGPAI receives a question:

```text
Are MAG sales up in Chicago?
```

But the tiny ANN does **not** process those words directly.

Instead, the sentence is represented as a numeric feature vector:

```text
[Sales Change, Location Chicago, Trend Up/Down, Question Type, Time Context, Overall Sentiment]
```

In a real LLM these would come from token embeddings and many layers.  
In this educational demo, we use six easy-to-understand features so we can see the mechanics clearly.


In [ ]:
# =============================================================================
# Module Name: tiny_nn_training_notebook_v1
# Author: javaboy-vk
# Date: 2026-06-01
# Version: 1.0
# Description:
#   Jupyter notebook demonstration of the MAGPAI tiny ANN training process.
#   The notebook shows how a fixed question is re-evaluated after six training
#   examples update the model's weights and bias.
# =============================================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Dict

import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output

# Notebook display settings
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 20)


## 2. Define the Fixed MAGPAI Question

The question below **never changes**.

This is the most important teaching point:

```text
Training does not change the question.
Training changes the weights and bias.
The changed weights and bias change the answer.
```


In [ ]:
FIXED_QUESTION = "Are MAG sales up in Chicago?"

FEATURE_NAMES = [
    "Sales Change",
    "Location Chicago",
    "Trend Up/Down",
    "Question Type",
    "Time Context",
    "Overall Sentiment",
]

FIXED_QUESTION_VECTOR = [0.81, 1.00, 1.00, 1.00, 0.60, 0.85]

display(Markdown(f"## Fixed Question\n\n> **{FIXED_QUESTION}**"))

pd.DataFrame(
    [FIXED_QUESTION_VECTOR],
    columns=FEATURE_NAMES,
    index=["Fixed Question Vector"]
)


## 3. Training Examples

These six examples are MAGPAI's **experience**.

Each example has:

- a training sentence,
- a numeric tensor/vector representation,
- a target label:
  - `1` means YES, sales are up,
  - `0` means NO, sales are not up.

The fixed question remains the same. These examples teach the model how to answer it better.


In [ ]:
@dataclass(frozen=True)
class TrainingExample:
    sentence: str
    target: int
    vector: List[float]


training_examples = [
    TrainingExample("Chicago sales increased by 15%", 1, [0.65, 1.00, 0.78, 1.00, 0.40, 0.70]),
    TrainingExample("Chicago sales increased by 22%", 1, [0.78, 1.00, 0.88, 1.00, 0.45, 0.78]),
    TrainingExample("Chicago sales decreased by 8%", 0, [0.28, 1.00, 0.18, 1.00, 0.35, 0.20]),
    TrainingExample("Chicago sales increased by 18%", 1, [0.70, 1.00, 0.82, 1.00, 0.50, 0.74]),
    TrainingExample("Chicago sales increased by 30%", 1, [0.92, 1.00, 1.00, 1.00, 0.55, 0.90]),
    TrainingExample("Chicago sales increased by 27%", 1, [0.81, 1.00, 1.00, 1.00, 0.60, 0.85]),
]

rows = []
for i, example in enumerate(training_examples, start=1):
    row = {
        "Step": i,
        "Training Sentence": example.sentence,
        "Target": example.target,
        "Target Meaning": "YES" if example.target == 1 else "NO",
    }
    row.update(dict(zip(FEATURE_NAMES, example.vector)))
    rows.append(row)

training_df = pd.DataFrame(rows)
training_df


## 4. Define the Tiny ANN

This tiny ANN has one neuron:

```text
z = w·x + b
prediction = sigmoid(z)
```

Where:

- `x` is the input vector,
- `w` is the weights vector,
- `b` is the bias,
- `z` is the raw score,
- `sigmoid(z)` converts the raw score into a value between `0` and `1`.

This is deliberately small so we can inspect everything.


In [ ]:
def sigmoid(value: float) -> float:
    """Convert a raw ANN score into a probability-like value between 0 and 1."""
    return 1.0 / (1.0 + math.exp(-value))


def dot(left: List[float], right: List[float]) -> float:
    """Calculate the dot product between two vectors."""
    return sum(a * b for a, b in zip(left, right))


class MAGPAITinyANN:
    """
    A tiny educational ANN with one linear neuron.

    Conceptually:
        z = w·x + b
        prediction = sigmoid(z)

    During training:
        error = target - prediction
        weight = weight + learning_rate * error * input
        bias = bias + learning_rate * error
    """

    def __init__(self, weights: List[float], bias: float, learning_rate: float) -> None:
        self.weights = weights
        self.bias = bias
        self.learning_rate = learning_rate

    def raw_score(self, vector: List[float]) -> float:
        return dot(self.weights, vector) + self.bias

    def predict(self, vector: List[float]) -> float:
        return sigmoid(self.raw_score(vector))

    def train_one(self, example: TrainingExample) -> Dict[str, object]:
        prediction_before = self.predict(example.vector)
        raw_before = self.raw_score(example.vector)
        error = example.target - prediction_before

        old_weights = self.weights.copy()
        old_bias = self.bias

        for index, input_value in enumerate(example.vector):
            self.weights[index] += self.learning_rate * error * input_value

        self.bias += self.learning_rate * error

        return {
            "sentence": example.sentence,
            "target": example.target,
            "raw_before": raw_before,
            "prediction_before": prediction_before,
            "error": error,
            "old_weights": old_weights,
            "new_weights": self.weights.copy(),
            "old_bias": old_bias,
            "new_bias": self.bias,
        }


## 5. Initialize Weights and Bias

At the beginning, MAGPAI has not learned from these six examples yet.

The starting weights and bias are intentionally weak:

- small weights,
- negative bias,
- low initial confidence.

This makes the learning movement easier to see.


In [ ]:
initial_weights = [0.150, 0.200, 0.200, 0.200, 0.200, 0.150]
initial_bias = -1.100
learning_rate = 0.70

model = MAGPAITinyANN(
    weights=initial_weights.copy(),
    bias=initial_bias,
    learning_rate=learning_rate,
)

initial_prediction = model.predict(FIXED_QUESTION_VECTOR)

display(Markdown(f"### Initial prediction for fixed question: `{initial_prediction:.3f}`"))

pd.DataFrame(
    {
        "Feature": FEATURE_NAMES,
        "Initial Weight": initial_weights,
    }
)


## 6. Visualization Helpers

These functions draw:

1. the current tensor/feature vector,
2. the current weights,
3. the prediction history after each training example.

The notebook uses normal Matplotlib so it can run anywhere Jupyter runs.


In [ ]:
def display_vector_table(vector: List[float], title: str) -> None:
    df = pd.DataFrame([vector], columns=FEATURE_NAMES, index=[title])
    display(df)


def plot_feature_vector(vector: List[float], title: str) -> None:
    plt.figure(figsize=(10, 4))
    plt.bar(FEATURE_NAMES, vector)
    plt.ylim(0, 1.05)
    plt.title(title)
    plt.ylabel("Feature Strength")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_weights(weights: List[float], title: str) -> None:
    plt.figure(figsize=(10, 4))
    plt.bar(FEATURE_NAMES, weights)
    plt.title(title)
    plt.ylabel("Learned Weight")
    plt.xticks(rotation=25, ha="right")
    plt.axhline(0, linewidth=1)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_prediction_history(history: List[float]) -> None:
    labels = ["Step 0"] + [f"Step {i}" for i in range(1, len(history))]
    plt.figure(figsize=(10, 4))
    plt.plot(labels, history, marker="o")
    plt.ylim(0, 1.05)
    plt.title('Prediction History for Fixed Question: "Are MAG sales up in Chicago?"')
    plt.ylabel("Prediction Probability")
    plt.grid(axis="y", alpha=0.3)
    for x, y in zip(labels, history):
        plt.text(x, y + 0.035, f"{y:.2f}", ha="center")
    plt.show()


## 7. Train Step by Step

Run the next cell.

It processes all six training examples one by one and records:

- current training sentence,
- tensor/vector values,
- prediction before update,
- error,
- weight changes,
- bias change,
- re-answer to the fixed MAGPAI question.


In [ ]:
# Reinitialize so this cell can be safely run multiple times.
model = MAGPAITinyANN(
    weights=initial_weights.copy(),
    bias=initial_bias,
    learning_rate=learning_rate,
)

prediction_history = [model.predict(FIXED_QUESTION_VECTOR)]
training_log = []

for step, example in enumerate(training_examples, start=1):
    update_info = model.train_one(example)
    fixed_question_prediction = model.predict(FIXED_QUESTION_VECTOR)
    prediction_history.append(fixed_question_prediction)

    training_log.append({
        "Step": step,
        "Training Sentence": example.sentence,
        "Target": example.target,
        "Prediction Before Update": update_info["prediction_before"],
        "Error": update_info["error"],
        "Bias Before": update_info["old_bias"],
        "Bias After": update_info["new_bias"],
        "Fixed Question Prediction After Update": fixed_question_prediction,
    })

training_log_df = pd.DataFrame(training_log)
training_log_df


## 8. Inspect Each Training Step

Change `step_to_inspect` from `1` to `6` and re-run the cell.

This lets you slow down and inspect one training example at a time.


In [ ]:
step_to_inspect = 6  # Change this from 1 to 6.

# Re-run training up to the selected step so we can inspect the exact state.
model = MAGPAITinyANN(
    weights=initial_weights.copy(),
    bias=initial_bias,
    learning_rate=learning_rate,
)

history = [model.predict(FIXED_QUESTION_VECTOR)]
selected_info = None
selected_example = None

for step, example in enumerate(training_examples, start=1):
    info = model.train_one(example)
    history.append(model.predict(FIXED_QUESTION_VECTOR))

    if step == step_to_inspect:
        selected_info = info
        selected_example = example
        break

display(Markdown(f"""## Inspecting Step {step_to_inspect}

**Training sentence:** {selected_example.sentence}  
**Target:** {selected_example.target} ({'YES' if selected_example.target == 1 else 'NO'})  

### What happened?

MAGPAI made a prediction, compared it to the target, calculated the error, then updated the weights and bias.

- Prediction before update: `{selected_info['prediction_before']:.3f}`
- Error: `{selected_info['error']:.3f}`
- Bias before: `{selected_info['old_bias']:.3f}`
- Bias after: `{selected_info['new_bias']:.3f}`
- Fixed question prediction after this step: `{history[-1]:.3f}`
"""))

display_vector_table(selected_example.vector, f"Step {step_to_inspect} Tensor / Feature Vector")
plot_feature_vector(selected_example.vector, f"Step {step_to_inspect}: Tensor / Graphics Vector")

weights_df = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "Weight Before": selected_info["old_weights"],
    "Weight After": selected_info["new_weights"],
    "Change": [
        after - before
        for before, after in zip(selected_info["old_weights"], selected_info["new_weights"])
    ],
})
display(weights_df)
plot_weights(selected_info["new_weights"], f"Step {step_to_inspect}: Learned Weights After Update")

plot_prediction_history(history)


## 9. Final State After All Six Training Examples

This cell reproduces the final state that corresponds to the browser demo final screen.


In [ ]:
# Re-train through all six examples.
model = MAGPAITinyANN(
    weights=initial_weights.copy(),
    bias=initial_bias,
    learning_rate=learning_rate,
)

prediction_history = [model.predict(FIXED_QUESTION_VECTOR)]

for example in training_examples:
    model.train_one(example)
    prediction_history.append(model.predict(FIXED_QUESTION_VECTOR))

final_prediction = prediction_history[-1]
final_answer = "YES" if final_prediction >= 0.5 else "NO"

display(Markdown(f"""# Final MAGPAI Answer

Fixed question:

> **{FIXED_QUESTION}**

Final prediction probability:

> **{final_prediction:.3f}**

Final answer:

> **{final_answer}**
"""))

final_weights_df = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "Initial Weight": initial_weights,
    "Final Weight": model.weights,
    "Change": [final - start for start, final in zip(initial_weights, model.weights)],
})

display(final_weights_df)

display(Markdown(f"""### Final Bias

- Initial bias: `{initial_bias:.3f}`
- Final bias: `{model.bias:.3f}`
- Bias change: `{model.bias - initial_bias:+.3f}`
"""))

plot_weights(model.weights, "Final Learned Weights After Six Training Examples")
plot_prediction_history(prediction_history)


## 10. What This Demonstrates

The tiny ANN is not doing magic.

It is doing this:

```text
input vector
  ↓
weighted sum
  ↓
add bias
  ↓
sigmoid
  ↓
prediction
```

Training means:

```text
compare prediction to target
  ↓
calculate error
  ↓
adjust weights and bias
```

For MAGPAI:

```text
Tensor / Vector = evidence
Weights = learned importance
Bias = default tendency
Prediction = current answer
Training = changing weights and bias from examples
```

That is the essence of the machine learning concept we are trying to teach.


## Optional Lab Exercise

Try changing these values and re-running the notebook:

1. Change the learning rate from `0.70` to `0.20`.
2. Make Step 3 more negative by changing its vector.
3. Add a seventh training example.
4. Change the initial bias from `-1.100` to `0.000`.
5. Observe how the final answer changes.

The goal is to see that the model's answer is controlled by the learned weights and bias.
